# TAVI 분석 코드 설명서
**Thermal Accessibility Vulnerability Index — 코드 해설**

---

- 작성자: jinshingeo
- 날짜: 2026-04-16
- 버전: TAVI_v2
- 대상: 코드 구조 및 방법론 이해가 필요한 독자

---

## 0. 연구 개요

### 핵심 질문
- 폭염 시 열환경이 보행 접근성을 얼마나 제한하는가
- 어느 역이 열환경에 취약한가

### 분석 지역 및 대상
- 지역: 서울 성동구
- 대상: 지하철역 7개 (왕십리·행당·응봉·뚝섬·성수·서울숲·옥수)
- 기준 기간: 2025년 7월 28일 ~ 8월 3일 (7일 폭염 평균)

### 분석 흐름

```
[Step 1] OSM 보행 네트워크 구축
            ↓
[Step 2] SVF(Sky View Factor) 계산
            ↓
[Step 3] SOLWEIG 기반 UTCI 계산 (링크별·시간대별)
            ↓
[Step 4] Classic Catchment vs Thermal Catchment 비교
            ↓
[Step 5] reduction_pct 산출 및 공간 시각화
```

---

## 1. 핵심 파라미터

| 파라미터 | 값 | 근거 |
|---------|-----|------|
| `WALK_SPEED` | 4.5 km/h | 성인 평균 보행속도 문헌 범위 내 |
| `TIME_BUDGET` | 15분 (900초) | Moreno et al. (2021) 15분 도시 |
| `UTCI_THRESHOLD` | 38°C | Bröde et al. (2012) "very strong heat stress" 시작점 |
| `CANOPY_COEFF` | 2.5°C | Chen & Ng (2012) 캐노피 UTCI 감소 효과 |

### 민감도 분석 예정 항목
- `TIME_BUDGET`: 10 / 15 / 20분
- `UTCI_THRESHOLD`: 35 / 38 / 41°C

---

## Step 1. 보행 네트워크 구축
**파일**: `01_네트워크/01_download_network.py`

### 목적
- OSM(OpenStreetMap)에서 성동구 보행 도로 네트워크 다운로드
- 각 링크(도로 선분)에 `length`(m) 속성 포함

### 핵심 개념
- 노드(Node): 교차점 또는 도로 끝점
- 엣지/링크(Edge): 두 노드를 잇는 도로 선분
- 무방향 그래프(Undirected): 양방향 통행 가정

In [ ]:
import osmnx as ox
import networkx as nx

# 성동구 보행 네트워크 다운로드
G = ox.graph_from_place('성동구, 서울특별시, 대한민국',
                         network_type='walk')

# 무방향 그래프로 변환 (양방향 보행 가정)
G = G.to_undirected()

# GeoDataFrame으로 변환 (시각화 및 공간 연산용)
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

print(f'노드 수: {len(nodes_gdf):,}')
print(f'링크 수: {len(edges_gdf):,}')

---

## Step 2. SVF(Sky View Factor) 계산
**파일**: `04_분석결과/15_svf_per_link.py`

### SVF란
- Sky View Factor: 특정 지점에서 하늘이 보이는 비율 (0~1)
- SVF = 1.0 : 완전 개활지 (교량, 공원) — 태양복사 최대 노출
- SVF = 0.3 : 고층 빌딩 협곡 — 태양복사 차단

### 계산 공식 — Oke (1987) H/W Street Canyon

$$\text{SVF} = \frac{1}{\sqrt{1 + (H/W)^2}}$$

- $H$ = 링크 주변 20m 내 건물 평균 높이 (층수 × 3m)
- $W$ = 도로 유형별 표준 폭 (국토부 기준)

### 추가 계산: 가로수 캐노피 비율
- 링크 주변 15m 버퍼 내 가로수 면적 / 버퍼 전체 면적
- 출력: `link_svf_canopy.csv`

In [ ]:
import numpy as np

# Oke (1987) H/W Street Canyon 공식
def calc_svf(H, W):
    """
    H: 링크 주변 건물 평균 높이 (m)
    W: 도로폭 (m)
    반환: SVF (0~1)
    """
    return 1.0 / np.sqrt(1.0 + (H / W) ** 2)

# 예시
print('개활지(H=0, W=10):', calc_svf(0.001, 10))     # ≈ 1.0
print('일반주거(H=9, W=9):', round(calc_svf(9, 9), 3)) # ≈ 0.707
print('밀집협곡(H=30, W=6):', round(calc_svf(30, 6), 3)) # ≈ 0.196

---

## Step 3. SOLWEIG 기반 UTCI 계산
**파일**: `04_분석결과/19_solweig_utci.py`

### UTCI란
- Universal Thermal Climate Index
- 기온·복사열·습도·풍속을 통합한 인체 열스트레스 지표
- Bröde et al. (2012) 국제 표준

### 계산 파이프라인

```
Open-Meteo API (기온, 습도, 풍속, 일사량)
            +
        SVF (링크별)
            ↓
        MRT 계산 (Lindberg et al. 2008)
            ↓
   pythermalcomfort → UTCI
            ↓
   캐노피 보정 (Chen & Ng 2012)
            ↓
       utci_final (링크별·시간별)
```

### MRT 계산 — SOLWEIG 단순화 (Lindberg et al. 2008)

**주간 (GHI > 10 W/m²)**:
$$\text{MRT} = T_{air} + 0.5 \times \sqrt{\text{GHI} \times \cos_z} \times \text{SVF}$$

**야간**:
$$\text{MRT} = T_{air} - 2.0 \times \text{SVF}$$

- SVF = 1 (개활지): 태양복사 최대 → MRT 최고
- SVF = 0 (협곡): 태양복사 차단 → MRT ≈ 기온

### 캐노피 보정
$$\text{UTCI}_{final} = \text{UTCI}_{solweig} - 2.5 \times \text{canopy\_ratio} \times \text{solar\_factor}$$

- 가로수가 많을수록 UTCI 감소 (최대 2.5°C)
- 정오(12~13시)에 solar_factor = 1.0 → 보정 효과 최대

In [ ]:
from pythermalcomfort.models import utci

def compute_mrt(Tair, GHI, svf, cos_z):
    """SOLWEIG 단순화 MRT 계산"""
    if GHI > 10:   # 주간
        delta = 0.5 * np.sqrt(GHI * cos_z)
        return Tair + delta * svf
    else:          # 야간
        return Tair - 2.0 * svf

# 예시: 13시, 개활지(SVF=1.0) vs 협곡(SVF=0.3)
Tair, GHI, cos_z = 35.0, 700, 0.85

mrt_open   = compute_mrt(Tair, GHI, svf=1.0, cos_z=cos_z)
mrt_canyon = compute_mrt(Tair, GHI, svf=0.3, cos_z=cos_z)

print(f'개활지 MRT: {mrt_open:.1f}°C')
print(f'협곡  MRT: {mrt_canyon:.1f}°C')
print(f'차이: {mrt_open - mrt_canyon:.1f}°C')

### UTCI 열스트레스 카테고리 (Bröde et al. 2012)

| UTCI 범위 | 카테고리 | 보행 영향 |
|-----------|---------|----------|
| < 26°C | 쾌적 | 없음 |
| 26 ~ 32°C | 약한 열스트레스 | 경미 |
| 32 ~ 38°C | 강한 열스트레스 | 주의 필요 |
| **38 ~ 46°C** | **매우 강한 열스트레스** | **보행 회피 기준 (임계값)** |
| > 46°C | 극한 열스트레스 | 위험 |

**38°C를 Thermal Catchment의 하드 컷 기준으로 사용**

---

## Step 4. Classic vs Thermal Catchment
**파일**: `04_분석결과/20_catchment_solweig.py`

### Catchment란
- 특정 역에서 시간예산(15분) 내 도보로 접근 가능한 구역
- Space-Time Prism(STP) 이론에서 파생
- 역에서 출발하는 역방향 다익스트라(Dijkstra) 알고리즘으로 계산

---

### Classic vs Thermal — 핵심 차이 비교

| 구분 | Classic Catchment | Thermal Catchment |
|------|------------------|-----------------|
| 그래프 | 전체 보행 네트워크 | UTCI < 38°C 링크만 |
| 이동비용 | `length / WALK_SPEED` | 동일 (단, 뜨거운 링크 없음) |
| 이론 근거 | STP 기본 모델 | Bröde et al. (2012) 임계값 |
| 결과 | 잠재적 접근 구역 | 폭염 시 실제 접근 가능 구역 |

---

### 알고리즘 흐름

```
[Classic]
전체 그래프 G
    ↓
Dijkstra(G, 역_노드, 시간예산=900초)
    ↓
classic_nodes = {n | 이동시간 ≤ 900초}


[Thermal]
전체 그래프 G
    ↓
UTCI ≥ 38°C 링크 제거 → G_thermal
    ↓
Dijkstra(G_thermal, 역_노드, 시간예산=900초)
    ↓
thermal_nodes = {n | 뜨거운 링크 피해서 이동시간 ≤ 900초}


[차이]
lost_nodes    = classic_nodes - thermal_nodes
reduction_pct = |lost_nodes| / |classic_nodes| × 100
```

In [ ]:
import networkx as nx

WALK_SPEED  = 4.5 * 1000 / 3600   # m/s (= 1.25 m/s)
TIME_BUDGET = 15 * 60              # 초 (= 900초)
THRESHOLD   = 38.0                 # °C (Bröde et al. 2012)


def compute_catchment(G, station_node, hot_edges_set):
    """
    Classic / Thermal Catchment 동시 계산

    Parameters
    ----------
    G              : 전체 보행 네트워크 (networkx Graph)
    station_node   : 역 노드 ID
    hot_edges_set  : UTCI >= 38°C 인 (u, v) 튜플 집합

    Returns
    -------
    dict : classic_nodes, thermal_nodes, lost_nodes, reduction_pct
    """

    # --- 1. 이동시간 엣지 속성 설정 ---
    for u, v, data in G.edges(data=True):
        data['travel_time'] = data.get('length', 0) / WALK_SPEED

    # --- 2. Classic Catchment ---
    # 전체 그래프에서 Dijkstra 실행
    classic_dist = nx.single_source_dijkstra_path_length(
        G, station_node, cutoff=TIME_BUDGET, weight='travel_time'
    )
    classic_nodes = set(classic_dist.keys())

    # --- 3. Thermal Catchment ---
    # 뜨거운 링크 제거
    G_thermal = G.copy()
    hot_list  = [
        (u, v) for u, v in G_thermal.edges()
        if (str(u), str(v)) in hot_edges_set
        or (str(v), str(u)) in hot_edges_set
    ]
    G_thermal.remove_edges_from(hot_list)

    # 수정된 그래프에서 동일한 Dijkstra 실행
    thermal_dist  = nx.single_source_dijkstra_path_length(
        G_thermal, station_node, cutoff=TIME_BUDGET, weight='travel_time'
    )
    thermal_nodes = set(thermal_dist.keys())

    # --- 4. 차집합 계산 ---
    lost_nodes    = classic_nodes - thermal_nodes
    reduction_pct = round(len(lost_nodes) / max(len(classic_nodes), 1) * 100, 1)

    return {
        'classic_nodes':  classic_nodes,
        'thermal_nodes':  thermal_nodes,
        'lost_nodes':     lost_nodes,
        'classic_count':  len(classic_nodes),
        'thermal_count':  len(thermal_nodes),
        'lost_count':     len(lost_nodes),
        'reduction_pct':  reduction_pct,
    }

### 시각화 색상 체계

```
초록 링크  →  thermal_nodes 내부 링크
              "폭염에도 접근 가능한 구역"

빨강 링크  →  lost_nodes 내부 링크
              "폭염 전에는 접근 가능했으나 이제는 불가능한 구역"
              = 열환경으로 인한 접근성 상실 구역

회색 링크  →  catchment 외부
              "15분 이내 도달 불가 (열환경 무관)"
```

$$\text{reduction\_pct} = \frac{|\text{빨강}|}{|\text{초록}| + |\text{빨강}|} \times 100 \; (\%)$$

---

## Step 5. 결과 해석

### 성동구 7개 역 결과 (SOLWEIG 기반, 13시)

| 역 | reduction_pct | 해석 |
|----|--------------|------|
| 응봉역 | 68.6% | 최고 취약 — 하천변, 개활지 多, 그늘 부족 |
| 서울숲역 | 67.8% | 서울숲 인접에도 높은 취약성 |
| 왕십리역 | 46.1% | 중간 — 대형 환승역 |
| 행당역 | 41.1% | 중간 |
| 뚝섬역 | 35.6% | 비교적 낮음 |
| 옥수역 | 29.8% | 낮음 — 한강 인접, 상대적 그늘 |
| 성수역 | 13.0% | 최저 취약 — 하천+녹지 완충 효과 |

### 핵심 발견
- 동일 성동구 내에서도 응봉역(68.6%)과 성수역(13.0%)은 5배 이상 차이
- 단순 거리 기반 Catchment는 이 차이를 전혀 포착하지 못함
- 열환경 반영 여부가 접근성 평가에 결정적 영향

---

## Step 6. TAVI 프레임워크 (진행 중)

### H×E×V 구조 (IPCC AR6 기반)

$$\text{TAVI}(\text{역}) = \underbrace{\text{reduction\_pct}}_{H \times E} \times \underbrace{\text{vulnerability\_ratio}}_{V}$$

| 요소 | 내용 | 데이터 | 상태 |
|------|------|--------|------|
| H (Hazard) | UTCI 열환경 | SOLWEIG 기반 | 완료 |
| E (Exposure) | Thermal Catchment reduction_pct | Dijkstra | 완료 |
| V (Vulnerability) | (65세+ + 14세-) / 전체 인구 | 통계청 SGIS | **수집 예정** |

### Kar et al. (2023) 연결

```
Kar et al.                        TAVI
─────────────────────────────────────────────
Hard constraint                   H × E
(물리적 이동 제약)                  (UTCI 열환경 장벽)

Soft constraint                   V
(개인 인식·선호 제약)               (취약 집단일수록 열 패널티 강화)
```

### 현재 구현 상태
- `20_catchment_solweig.py`: H × E 완료
- V 컴포넌트: 통계청 SGIS 데이터 수집 후 구현 예정

---

## Step 7. V 컴포넌트 — 취약인구 비율
**파일**: `04_분석결과/25_vulnerability_component.py`

### 개념
- V (Vulnerability) = 역 Catchment 내 취약인구 비율
- 취약인구 = 열에 민감하면서 대중교통 의존도가 높은 집단

$$V = \frac{\text{65세 이상 인구} + \text{14세 이하 인구}}{\text{전체 인구}}$$

### 왜 이 집단인가

| 집단 | 열 취약 이유 | 이동성 이유 |
|------|------------|------------|
| 65세 이상 | 체온 조절 능력 저하, 만성질환 동반, 폭염 사망률 높음 | 자가용 운전 불가 비율 높음 |
| 14세 이하 | 체표면적 대비 열 흡수 비율 높음, 스스로 회피 어려움 | 독립 이동 불가 |

두 집단 모두 **대중교통 의존도가 높고**, **폭염 시 스스로 회피 결정이 어렵다**는 공통점이 있음.

---

### 계산 파이프라인

```
[1] OSM 네트워크에서 Classic Catchment 노드셋 추출
        ↓
[2] 노드 좌표 → Convex Hull → Catchment 폴리곤 (EPSG:5186)
        ↓
[3] 행정동 경계 (SGIS) × Catchment 폴리곤 — 공간 교차
        ↓
[4] 교차 면적 / 행정동 전체 면적 = 면적 가중치
        ↓
[5] vulnerability_ratio = 가중 평균(V) per 역
```

### 공간 가중 평균의 이유
- 하나의 Catchment가 여러 행정동에 걸쳐 있음
- 완전히 포함된 행정동 = 가중치 1.0
- 일부만 포함된 행정동 = 가중치 < 1.0 (면적 비례)
- 행정동 인구를 그대로 사용하면 과대/과소 추정 발생 → 면적 가중 보정

---

### 데이터 수집 방법 (수동)

```
통계청 SGIS (sgis.kostat.go.kr)
  → 통계지도서비스
  → 행정구역별 통계
  → 서울시 > 성동구 > 행정동
  → 연령별 인구 (5세 단계)
  → CSV 다운로드

파일 저장 위치:
  02_기상데이터/seongdong_dong_population.csv
  02_기상데이터/seongdong_dong_boundary.geojson
```

> SGIS 데이터 수집 전까지는 Mock 데이터로 파이프라인 실행 가능

In [ ]:
from shapely.geometry import MultiPoint
import geopandas as gpd

# ── Catchment 폴리곤 생성 ──────────────────────────────────────────────
def make_catchment_polygon(G, station_node, time_budget=900, walk_speed=1.25):
    """
    Classic Catchment 노드셋 → Convex Hull 폴리곤

    - Dijkstra로 time_budget 내 도달 가능한 모든 노드 탐색
    - 해당 노드들의 WGS84 좌표로 Convex Hull 생성
    - 반환: shapely Polygon (EPSG:4326)
    """
    dist = nx.single_source_dijkstra_path_length(
        G, station_node, cutoff=time_budget, weight='travel_time'
    )
    coords = [(G.nodes[n]['x'], G.nodes[n]['y']) for n in dist if n in G.nodes]
    if len(coords) < 3:
        return None
    return MultiPoint(coords).convex_hull


# ── 면적 가중 V 계산 ────────────────────────────────────────────────────
def compute_weighted_vulnerability(catchment_poly_5186, dong_gdf_5186, pop_df):
    """
    면적 가중 취약인구 비율 계산

    Parameters
    ----------
    catchment_poly_5186 : shapely Polygon (EPSG:5186)
    dong_gdf_5186       : 행정동 경계 GeoDataFrame (EPSG:5186)
    pop_df              : 인구 데이터 DataFrame (dong_name, total_pop, elderly_pop, children_pop)

    Returns
    -------
    float : vulnerability_ratio
    """
    rows = []
    for _, dong_row in dong_gdf_5186.merge(pop_df, on='dong_name').iterrows():
        intersect = catchment_poly_5186.intersection(dong_row.geometry)
        if intersect.is_empty:
            continue
        weight = intersect.area / dong_row.geometry.area          # 면적 비율 가중치
        rows.append({
            'dong':     dong_row['dong_name'],
            'weight':   weight,
            'total':    dong_row['total_pop']    * weight,
            'elderly':  dong_row['elderly_pop']  * weight,
            'children': dong_row['children_pop'] * weight,
        })

    if not rows:
        return float('nan')

    import pandas as pd
    df = pd.DataFrame(rows)
    total = df['total'].sum()
    vuln  = df['elderly'].sum() + df['children'].sum()
    return round(vuln / total, 4) if total > 0 else float('nan')


# 예시 실행 흐름 (실제 데이터 수집 후 실행)
# catchment_poly = make_catchment_polygon(G, station_node)
# catchment_5186 = gpd.GeoSeries([catchment_poly], crs='EPSG:4326').to_crs(5186).iloc[0]
# v = compute_weighted_vulnerability(catchment_5186, dong_gdf, pop_df)
print("V 컴포넌트 함수 정의 완료")

---

## Step 8. TAVI 지수 최종 산출
**파일**: `04_분석결과/26_tavi_index.py`

### 공식

$$\text{TAVI}(\text{역}) = \text{reduction\_pct} \times \text{vulnerability\_ratio}$$

- reduction_pct: Step 4에서 산출 (H×E, 단위 %)
- vulnerability_ratio: Step 7에서 산출 (V, 단위 0~1)
- TAVI 단위: % × (무차원) = %로 해석 가능

### 예시 계산

| 역 | reduction_pct (E) | vulnerability_ratio (V) | TAVI |
|----|------------------|------------------------|------|
| 응봉역 | 68.6% | 0.26 | 17.8 |
| 서울숲역 | 67.8% | 0.24 | 16.3 |
| 왕십리역 | 46.1% | 0.23 | 10.6 |
| 성수역 | 13.0% | 0.25 | 3.3 |

> 위 V 수치는 Mock 데이터 기준 예시 — SGIS 수집 후 갱신 필요

---

### 해석
- **TAVI 높음**: 폭염 시 도보 접근성이 크게 떨어지는(E↑) 지역에 취약 인구가 집중(V↑)
  → 폭염 적응 정책의 **최우선 투자 대상**
- **TAVI 낮음**: E↓ 또는 V↓ 중 하나라도 낮으면 위험 감소
  - 성수역: E=13% (그늘 충분) → TAVI 낮음
  - 응봉역: E=68.6% (그늘 부족) + V↑ → TAVI 최고

---

### Kar et al. (2023) 프레임워크와의 연결

```
Kar et al.                        TAVI (본 연구)
──────────────────────────────────────────────────────
Hard constraint                   H × E
(물리적 이동 불가)                  (UTCI ≥ 38°C 링크 제거 → Catchment 감소)

Soft constraint                   V
(개인 인식·선호 기반 제약)           (취약 집단은 동일 열환경에서 더 큰 제약)
```

**차이점**: Kar et al.의 Soft constraint는 개인 설문 기반(주관적)이며,
TAVI의 V는 인구 센서스 기반(객관적)임. 후속 연구에서 설문 기반 Soft constraint 추가 가능.

In [ ]:
import json
import numpy as np

# ── TAVI 계산 핵심 로직 ────────────────────────────────────────────────
# 입력: catchment_solweig_summary.json + vulnerability_component.json

def compute_tavi(reduction_pct, vulnerability_ratio):
    """
    TAVI = reduction_pct(%) × vulnerability_ratio(0~1)

    Parameters
    ----------
    reduction_pct       : float, 0~100 (%)
    vulnerability_ratio : float, 0~1

    Returns
    -------
    float : TAVI 값
    """
    if np.isnan(reduction_pct) or np.isnan(vulnerability_ratio):
        return float('nan')
    return round(reduction_pct * vulnerability_ratio, 4)


# Mock 실행 예시
mock_data = {
    '응봉역':   {'reduction_pct': 68.6, 'vulnerability_ratio': 0.260},
    '서울숲역': {'reduction_pct': 67.8, 'vulnerability_ratio': 0.242},
    '왕십리역': {'reduction_pct': 46.1, 'vulnerability_ratio': 0.230},
    '행당역':   {'reduction_pct': 41.1, 'vulnerability_ratio': 0.235},
    '뚝섬역':   {'reduction_pct': 35.6, 'vulnerability_ratio': 0.228},
    '옥수역':   {'reduction_pct': 29.8, 'vulnerability_ratio': 0.240},
    '성수역':   {'reduction_pct': 13.0, 'vulnerability_ratio': 0.248},
}

print(f"{'역':<10} {'E (reduction)':>14} {'V (vulnerability)':>18} {'TAVI':>8}")
print("-" * 56)
for stn, vals in sorted(mock_data.items(),
                         key=lambda x: -x[1]['reduction_pct']):
    tavi = compute_tavi(vals['reduction_pct'], vals['vulnerability_ratio'])
    print(f"{stn:<10} {vals['reduction_pct']:>13.1f}%"
          f"  {vals['vulnerability_ratio']:>16.3f}"
          f"  {tavi:>8.2f}")

print("\n해석: TAVI 높을수록 폭염 정책 우선 투자 필요")
print("      (단, 위 V값은 Mock — SGIS 수집 후 재산출 필요)")

---

## Step 9. 링크 임계성 분석 — Micro Scale
**파일**: `04_분석결과/27_link_criticality.py`

### 왜 필요한가

TAVI(역)는 "어느 역이 취약한가"를 알려주지만,
**"그 역 안에서 어느 도로를 개선해야 하는가"**는 알려주지 않는다.

링크 임계성은 같은 프레임워크 안에서 스케일을 한 단계 낮춘 분석이다.

```
Macro Scale (역 단위)      TAVI(역) = reduction_pct × V
                               ↓
Micro Scale (링크 단위)    link_criticality(링크) = recovered_nodes / classic_nodes × 100
```

---

### 핵심 개념

hot link(UTCI ≥ 38°C로 제거된 링크) 하나를 복원했을 때,
얼마나 많은 노드가 접근성을 회복하는가.

$$\text{link\_criticality}(e) = \frac{|\text{nodes}_{G_{thermal} + e} - \text{thermal\_nodes}|}{|\text{classic\_nodes}|} \times 100 \; (\%)$$

- 값이 클수록: 이 링크 하나를 서늘하게 만들면 접근성 회복 효과 최대
- 값이 0: 이 링크를 복원해도 연결된 경로가 없어 효과 없음

---

### 알고리즘 흐름

```
[준비]
  G_thermal = G_base - hot_edges
  thermal_nodes = Dijkstra(G_thermal, 역, 15분)

[후보 링크 추림]
  hot_edges 중 thermal_nodes와 인접한 링크만 선별
  (내부 고립 링크는 단독 복원으로 접근성 회복 불가 → 스킵)
  → "경계 링크(gateway edge)"

[임계성 측정]
  for each gateway_edge (u, v):
      G_test = G_thermal + (u, v)
      new_nodes = Dijkstra(G_test, 역, 15분)
      recovered = new_nodes - thermal_nodes
      criticality = len(recovered) / len(classic_nodes) × 100
```

---

### 정책 함의

| 임계성 수준 | 의미 | 정책 제안 |
|-----------|------|----------|
| 높음 (>10%) | 이 링크 하나가 다수의 접근성을 차단 | 그늘막·가로수 긴급 설치 |
| 중간 (3~10%) | 복수 링크 함께 개선 시 효과적 | 인근 블록 통합 그린 인프라 |
| 낮음 (<3%) | 단독 효과 미미 | 장기 도시설계 과제로 분류 |

---

### 연구 프레임워크 완성 구조

```
TAVI Framework
│
├── Macro Scale
│     TAVI(역) = reduction_pct × V
│     → 투자 우선 역 선정
│     → "응봉역, 서울숲역 우선"
│
└── Micro Scale
      link_criticality(링크)
      → 구체적 개선 도로 제시
      → "응봉역 내 ○○로 X번 링크 우선 그늘막 설치"
```

이 두 스케일이 **동일한 데이터·동일한 모델**에서 파생됨 → 논문의 일관성 확보

In [ ]:
import networkx as nx

WALK_SPEED  = 4.5 * 1000 / 3600
TIME_BUDGET = 15 * 60
THRESHOLD   = 38.0


def compute_link_criticality(G_base, station_node, hot_edges_set, edge_time_dict):
    """
    각 hot edge의 접근성 회복 기여도(임계성) 계산

    Parameters
    ----------
    G_base          : 전체 보행 네트워크 (travel_time 속성 포함)
    station_node    : 역 노드 ID
    hot_edges_set   : UTCI >= 38°C 링크 (str(u), str(v)) 집합
    edge_time_dict  : {(str(u), str(v)): travel_time} 빠른 조회용

    Returns
    -------
    classic_nodes : set
    thermal_nodes : set
    criticality   : dict {(u, v): {'recovered_count': int, 'criticality_pct': float}}
    """

    # ── Classic Catchment ──────────────────────────────────────────────
    classic_dist  = nx.single_source_dijkstra_path_length(
        G_base, station_node, cutoff=TIME_BUDGET, weight='travel_time'
    )
    classic_nodes = set(classic_dist.keys())

    # ── Thermal Catchment ─────────────────────────────────────────────
    G_thermal = G_base.copy()
    hot_list  = [
        (u, v) for u, v in G_thermal.edges()
        if (str(u), str(v)) in hot_edges_set or (str(v), str(u)) in hot_edges_set
    ]
    G_thermal.remove_edges_from(hot_list)

    thermal_dist  = nx.single_source_dijkstra_path_length(
        G_thermal, station_node, cutoff=TIME_BUDGET, weight='travel_time'
    )
    thermal_nodes = set(thermal_dist.keys())
    lost_nodes    = classic_nodes - thermal_nodes

    if not lost_nodes:
        return classic_nodes, thermal_nodes, {}

    # ── 경계 링크(gateway edge) 필터 ──────────────────────────────────
    # thermal_nodes와 인접한 hot edge만 → 단독 복원 시 효과 있음
    candidate_edges = [
        (u, v) for u, v in hot_list
        if u in thermal_nodes or v in thermal_nodes
    ]

    # ── 링크별 임계성 측정 ─────────────────────────────────────────────
    criticality = {}
    for u, v in candidate_edges:
        t = edge_time_dict.get((str(u), str(v)),
            edge_time_dict.get((str(v), str(u)), 5.0))

        G_test    = G_thermal.copy()
        G_test.add_edge(u, v, travel_time=t)

        new_dist  = nx.single_source_dijkstra_path_length(
            G_test, station_node, cutoff=TIME_BUDGET, weight='travel_time'
        )
        recovered = set(new_dist.keys()) - thermal_nodes

        criticality[(u, v)] = {
            'recovered_count': len(recovered),
            'criticality_pct': round(len(recovered) / max(len(classic_nodes), 1) * 100, 2),
        }

    return classic_nodes, thermal_nodes, criticality


# 상위 링크 순위 출력 예시
print("compute_link_criticality() 정의 완료")
print()
print("해석 기준:")
print("  criticality_pct > 10%  → 단독 개선으로 접근성 큰 폭 회복 — 최우선 투자")
print("  criticality_pct 3~10%  → 인근 링크 묶음 개선 권장")
print("  criticality_pct < 3%   → 단독 효과 미미 — 장기 계획")

---

## 코드 파일 구조

```
04_분석결과/
  15_svf_per_link.py             SVF + 캐노피 계산 (Oke 1987)
  18_synthetic_dsm.py            합성 DSM 생성
  19_solweig_utci.py             SOLWEIG 기반 UTCI 계산
  20_catchment_solweig.py        [H×E] Classic vs Thermal Catchment
  22_regression_analysis.py      공간환경 변수 회귀분석 (Plan A)
  23_subway_preprocessing.py     지하철 승하차 전처리
  24_validation_analysis.py      검증 분석
  25_vulnerability_component.py  [V] 취약인구 비율 산출 (SGIS 기반)
  26_tavi_index.py               [TAVI] Macro Scale 지수 산출
  27_link_criticality.py         [Micro] 링크 임계성 — 개선 우선 도로 도출

전체 분석 흐름:
  19 → 20 (H×E)
       + 25 (V)
       → 26 (TAVI — Macro)
       → 27 (link_criticality — Micro)

프레임워크 해석:
  Macro: TAVI(역)        → 어느 역이 취약한가
  Micro: criticality(링크) → 어느 도로를 개선해야 하는가
```

---

## 참고문헌

1. **Bröde et al. (2012)** — UTCI 카테고리 및 38°C 임계값
   - Deriving the operational procedure for the Universal Thermal Climate Index (UTCI)
   - *International Journal of Biometeorology*, 56(3), 481–494

2. **Lindberg et al. (2008)** — SOLWEIG MRT 계산 원전
   - SOLWEIG 1.0 – Modelling spatial variations of 3D radiant fluxes
   - *International Journal of Biometeorology*, 52(7), 697–713

3. **Oke, T. R. (1987)** — SVF H/W Street Canyon 공식
   - *Boundary Layer Climates* (2nd ed.), Routledge

4. **Chen & Ng (2012)** — 가로수 캐노피 UTCI 감소 계수
   - Outdoor thermal comfort and outdoor activities
   - *Cities*, 29(2), 118–125

5. **Kar, Le & Miller (2023)** — Inclusive STP, Hard/Soft constraint
   - Inclusive Accessibility: Integrating Heterogeneous User Mobility Perceptions into Space-Time Prisms
   - *Annals of AAG*, 113(10), 2456–2479

6. **Moreno et al. (2021)** — 15분 도시 (TIME_BUDGET 근거)
   - Introducing the "15-Minute City"
   - *Smart Cities*, 4(1), 93–111

7. **IPCC AR6 (2021)** — Risk = H×E×V 표준 공식
   - Climate Change 2021: The Physical Science Basis